In [1]:
import google.generativeai as genai
import pandas as pd
from datasets import load_dataset
import os
from dotenv import load_dotenv
import time
from tqdm import tqdm
import numpy as np

d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.5-flash')
gemma_model = genai.GenerativeModel('gemma-3-1b-it')

In [34]:

def generate_gemma_response(prompt):
    response = gemma_model.generate_content(prompt,
                                          generation_config={
        'temperature': 0.7,
        'top_p': 0.95,
        'top_k': 40,
        'max_output_tokens': 8046,
        'stop_sequences': ['</answer>'],  # useful for your format
    }
    )
    return response.text

print("=" * 60)
print("Multi-Metric Domain-Aware Evaluation")
print("=" * 60)

# Define which domains have ground truth
GROUND_TRUTH_DOMAINS = {
    'math', 
    'commonsense_reasoning', 
    'science', 
    'numerical_reasoning', 
    'financial_reasoning', 
    'reading_comprehension'
}

CREATIVE_DOMAINS = {
    'code',
    'creative_ideation',
    'creative_writing',
    'summarization'
}


def parse_scores(result_text):
    '''Parse multiple scores from judge response'''
    scores = {}
    try:
        lines = result_text.split('\n')
        for line in lines:
            line_lower = line.lower()  # Convert to lowercase for matching
            
            if 'format accuracy:' in line_lower or 'format score:' in line_lower:
                score_str = line.split(':')[1].strip().split('/')[0].strip()
                scores['format'] = float(score_str)
            elif 'reasoning quality:' in line_lower or 'reasoning score:' in line_lower:
                score_str = line.split(':')[1].strip().split('/')[0].strip()
                scores['reasoning'] = float(score_str)
            elif 'answer quality:' in line_lower or 'quality score:' in line_lower:
                score_str = line.split(':')[1].strip().split('/')[0].strip()
                scores['answer_quality'] = float(score_str)
            elif 'answer accuracy:' in line_lower or 'accuracy score:' in line_lower:
                score_str = line.split(':')[1].strip().split('/')[0].strip()
                if score_str.upper() in ['N/A', 'N', 'NA', 'NONE']:
                    scores['answer_accuracy'] = None
                else:
                    scores['answer_accuracy'] = float(score_str)
        
        # Extract feedback
        feedback_start = result_text.find('Feedback:')
        if feedback_start != -1:
            scores['feedback'] = result_text[feedback_start:].replace('Feedback:', '').strip()
        else:
            scores['feedback'] = result_text
            
        return scores
    except Exception as e:
        print(f"Error parsing scores: {e}")
        print(f"Raw text: {result_text[:200]}")
        return None


def evaluate_with_ground_truth(input_text, model_response, ground_truth, domain):
    '''Evaluate response against ground truth with 4 separate scores'''
    
    prompt = f'''
Evaluate this AI model's response for a {domain} task with 4 separate scores:

Question/Task: {input_text}

Ground Truth Answer: {ground_truth}

Model Response: {model_response}

Provide 4 separate scores (each 1-10):

1. FORMAT ACCURACY (1-10):
   - Does response use <reasoning>...</reasoning> and <answer>...</answer> tags correctly?
   - Are tags properly nested and closed?
   - Is the structure clean and parseable?

2. REASONING QUALITY (1-10):
   - Is the reasoning trace clear and logical?
   - Does it show step-by-step thinking?
   - Are intermediate steps correct?
   - Is the reasoning helpful for understanding the solution?

3. ANSWER QUALITY (1-10):
   - Is the answer well-formatted and complete?
   - Is it clear and understandable?
   - Does it directly address the question?

4. ANSWER ACCURACY (0 = Wrong/1 =Correct):
   - Does the final answer match the ground truth?
   - Is it mathematically/factually correct?
   - If the answer is correct but is represented in a different format it's fine

Provide your evaluation in this exact format:
Format Accuracy: X/10
Reasoning Quality: X/10
Answer Quality: X/10
Answer Accuracy: 0/1
Feedback: [2-3 sentences explaining the scores]
'''
    
    try:
        response = gemini_model.generate_content(prompt)
        return parse_scores(response.text)
    except Exception as e:
        print(f"Error evaluating: {e}")
        return None


def evaluate_creative(input_text, model_response, domain):
    '''Evaluate creative/generative responses with 3 scores (no accuracy)'''
    
    domain_criteria = {
        'code': 'Code correctness, efficiency, handles edge cases, solves the problem',
        'creative_ideation': 'Creativity, relevance to prompt, feasibility, originality',
        'creative_writing': 'Writing quality, coherence, creativity, engagement',
        'summarization': 'Accuracy of main points, completeness, conciseness, clarity'
    }
    
    criteria = domain_criteria.get(domain, 'Quality, relevance, completeness, creativity')
    
    prompt = f'''
Evaluate this AI model's response for a {domain} task with 3 separate scores:

Task/Prompt: {input_text}

Model Response: {model_response}

Provide 3 separate scores (each 1-10):

1. FORMAT ACCURACY (1-10):
   - Does response use <reasoning>...</reasoning> and <answer>...</answer> tags correctly?
   - Are tags properly nested and closed?
   - Is the structure clean and parseable?

2. REASONING QUALITY (1-10):
   - Is the reasoning trace clear and logical?
   - Does it explain the thought process well?
   - Is the approach sound?
   - Does reasoning add value to understanding the solution?

3. ANSWER QUALITY (1-10):
   - {criteria}
   - Is the answer complete and well-executed?
   - Does it fulfill the prompt requirements?

Note: No accuracy score for {domain} as there's no single correct answer.

Provide your evaluation in this exact format:
Format Accuracy: X/10
Reasoning Quality: X/10
Answer Quality: X/10
Answer Accuracy: N/A
Feedback: [2-3 sentences explaining the scores]
'''
    
    try:
        response = gemini_model.generate_content(prompt)
        scores = parse_scores(response.text)
        if scores:
            scores['answer_accuracy'] = None  # N/A for creative domains
        return scores
    except Exception as e:
        print(f"Error evaluating: {e}")
        return None

def evaluate_model(eval_df):
    # Run multi-metric evaluation
    print("\nEvaluating responses with multi-metric scoring...")

    format_scores = []
    reasoning_scores = []
    answer_quality_scores = []
    answer_accuracy_scores = []
    feedbacks = []

    # has_output = 'output' in eval_df.columns

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Multi-metric judging"):
        # Skip failed generations
        # if row['error'] is not None:
        #     format_scores.append(0)
        #     reasoning_scores.append(0)
        #     answer_quality_scores.append(0)
        #     answer_accuracy_scores.append(0)
        #     feedbacks.append("Generation failed")
        #     continue
        
        domain = row['domain']
        
        # Determine evaluation strategy
        if domain in GROUND_TRUTH_DOMAINS:
            scores = evaluate_with_ground_truth(
                input_text=row['input'],
                model_response=row['model_response'],
                ground_truth=row['ground_truth'],
                domain=domain
            )
        elif domain in CREATIVE_DOMAINS:
            scores = evaluate_creative(
                input_text=row['input'],
                model_response=row['model_response'],
                domain=domain
            )
        else:
            scores = evaluate_creative(
                input_text=row['input'],
                model_response=row['model_response'],
                domain=domain
            )
        
        if scores:
            format_scores.append(scores.get('format', 0))
            reasoning_scores.append(scores.get('reasoning', 0))
            answer_quality_scores.append(scores.get('answer_quality', 0))
            answer_accuracy_scores.append(scores.get('answer_accuracy', 0) if scores.get('answer_accuracy') is not None else None)
            feedbacks.append(scores.get('feedback', 'Evaluation completed'))
        else:
            format_scores.append(0)
            reasoning_scores.append(0)
            answer_quality_scores.append(0)
            answer_accuracy_scores.append(None)
            feedbacks.append("Evaluation failed")
        
        # Rate limiting
        time.sleep(0.5)

    # Add to dataframe
    eval_df['format_accuracy'] = format_scores
    eval_df['reasoning_quality'] = reasoning_scores
    eval_df['answer_quality'] = answer_quality_scores
    eval_df['answer_accuracy'] = answer_accuracy_scores
    eval_df['judge_feedback'] = feedbacks

    # Calculate overall score (weighted average where applicable)
    overall_scores = []
    for i in range(len(eval_df)):
        if format_scores[i] == 0:  # Failed generation
            overall_scores.append(0)
        elif answer_accuracy_scores[i] is not None:  # Ground truth domains
            # Weighted: format 10%, reasoning 20%, quality 20%, accuracy 50%
            score = (format_scores[i] * 0.1 + 
                    reasoning_scores[i] * 0.2 + 
                    answer_quality_scores[i] * 0.2 + 
                    answer_accuracy_scores[i]*10 * 0.5)
            overall_scores.append(score)
        else:  # Creative domains
            # Weighted: format 15%, reasoning 35%, quality 50%
            score = (format_scores[i] * 0.15 + 
                    reasoning_scores[i] * 0.35 + 
                    answer_quality_scores[i] * 0.5)
            overall_scores.append(score)

    eval_df['overall_score'] = overall_scores

    # Calculate statistics
    print(f"\n✓ Multi-metric evaluation complete")
    print(f"\nOverall Statistics:")
    print(f"  Format Accuracy: {np.mean([s for s in format_scores if s > 0]):.2f}/10")
    print(f"  Reasoning Quality: {np.mean([s for s in reasoning_scores if s > 0]):.2f}/10")
    print(f"  Answer Quality: {np.mean([s for s in answer_quality_scores if s > 0]):.2f}/10")

    accuracy_valid = [s for s in answer_accuracy_scores if s is not None and s > 0]
    if accuracy_valid:
        print(f"  Answer Accuracy (verifiable domains): {np.mean(accuracy_valid):.2f}/10")

    print(f"  Overall Weighted Score: {np.mean([s for s in overall_scores if s > 0]):.2f}/10")

    # Per-domain statistics
    print("\nPer-Domain Scores:")
    for domain in eval_df['domain'].unique():
        domain_df = eval_df[eval_df['domain'] == domain]
        domain_overall = [s for s in domain_df['overall_score'] if s > 0]
        domain_format = [s for s in domain_df['format_accuracy'] if s > 0]
        domain_reasoning = [s for s in domain_df['reasoning_quality'] if s > 0]
        
        if domain_overall:
            print(f"\n  {domain}:")
            print(f"    Overall: {np.mean(domain_overall):.2f}/10")
            print(f"    Format: {np.mean(domain_format):.2f}/10")
            print(f"    Reasoning: {np.mean(domain_reasoning):.2f}/10")
            
            if domain in GROUND_TRUTH_DOMAINS:
                domain_acc = [s for s in domain_df['answer_accuracy'] if s is not None and s > 0]
                if domain_acc:
                    print(f"    Accuracy: {np.mean(domain_acc):.2f}/10")

    return eval_df

##General samples to test
def generate_general_eval_from_claude():
    eval_data = {
        'domain': [],
        'input': [],
        'output': [],
        'difficulty': []
    }

    # ============================================================================
    # MATH (10 questions from GSM8K-style)
    # ============================================================================
    math_questions = [
        {
            'input': 'A bakery sells 250 loaves of bread per day. Each loaf costs $3.50. If the bakery operates 6 days a week, how much revenue does it generate from bread sales in one week?',
            'output': '5250',
            'difficulty': 'easy'
        },
        {
            'input': 'Sarah has 3 times as many books as Tom. If Tom has 24 books, and Sarah gives away 1/4 of her books, how many books does Sarah have left?',
            'output': '54',
            'difficulty': 'medium'
        },
        {
            'input': 'A train travels at 75 mph for 2.5 hours, then at 60 mph for 1.5 hours. What is the total distance traveled?',
            'output': '277.5',
            'difficulty': 'easy'
        },
        {
            'input': 'If 5 workers can complete a job in 12 days, how many days would it take 8 workers to complete the same job, assuming they work at the same rate?',
            'output': '7.5',
            'difficulty': 'medium'
        },
        {
            'input': 'A rectangle has a length that is 3 meters longer than twice its width. If the perimeter is 54 meters, what is the area of the rectangle?',
            'output': '165',
            'difficulty': 'hard'
        },
        {
            'input': 'John bought 15 apples at $0.80 each and 20 oranges at $0.60 each. He paid with a $50 bill. How much change did he receive?',
            'output': '26',
            'difficulty': 'easy'
        },
        {
            'input': 'A store offers a 20% discount on an item, then adds 8% sales tax. If the original price is $125, what is the final price paid?',
            'output': '108',
            'difficulty': 'medium'
        },
        {
            'input': 'A swimming pool is being filled by two pipes. Pipe A can fill the pool in 6 hours, and Pipe B can fill it in 4 hours. If both pipes work together, how long will it take to fill the pool?',
            'output': '2.4',
            'difficulty': 'hard'
        },
        {
            'input': 'The sum of three consecutive even numbers is 126. What is the largest of these numbers?',
            'output': '44',
            'difficulty': 'medium'
        },
        {
            'input': 'A car depreciates by 15% each year. If it was worth $28,000 initially, what will be its value after 3 years?',
            'output': '17186.50',
            'difficulty': 'hard'
        }
    ]

    for q in math_questions:
        eval_data['domain'].append('math')
        eval_data['input'].append(q['input'])
        eval_data['output'].append(q['output'])
        eval_data['difficulty'].append(q['difficulty'])

    # ============================================================================
    # CODE (10 programming questions)
    # ============================================================================
    code_questions = [
        {
            'input': 'Write a Python function that takes a list of integers and returns the sum of all even numbers in the list.',
            'output': None,  # No single correct answer
            'difficulty': 'easy'
        },
        {
            'input': 'Write a function to check if a string is a palindrome (reads the same forwards and backwards), ignoring spaces and capitalization.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Implement a function that finds the second largest number in a list without sorting the entire list.',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Write a function that reverses the words in a sentence while maintaining the word order. For example, "hello world" becomes "olleh dlrow".',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Create a function that finds all pairs of numbers in a list that sum to a target value.',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Write a recursive function to calculate the nth Fibonacci number.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Implement a function that removes duplicate characters from a string while preserving order.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Write a function that merges two sorted lists into a single sorted list without using built-in sort functions.',
            'output': None,
            'difficulty': 'hard'
        },
        {
            'input': 'Create a function that validates if parentheses, brackets, and braces in a string are properly balanced.',
            'output': None,
            'difficulty': 'hard'
        },
        {
            'input': 'Write a function that rotates a matrix 90 degrees clockwise in-place.',
            'output': None,
            'difficulty': 'hard'
        }
    ]

    for q in code_questions:
        eval_data['domain'].append('code')
        eval_data['input'].append(q['input'])
        eval_data['output'].append(q['output'])
        eval_data['difficulty'].append(q['difficulty'])

    # ============================================================================
    # SCIENCE (10 questions)
    # ============================================================================
    science_questions = [
        {
            'input': 'What is the process by which plants convert sunlight into chemical energy?',
            'output': 'Photosynthesis',
            'difficulty': 'easy'
        },
        {
            'input': 'If an object has a mass of 50 kg and is accelerating at 2 m/s², what force is acting on it? (Use F=ma)',
            'output': '100',
            'difficulty': 'easy'
        },
        {
            'input': 'What is the pH value of a neutral solution at 25°C?',
            'output': '7',
            'difficulty': 'easy'
        },
        {
            'input': 'How many valence electrons does a carbon atom have?',
            'output': '4',
            'difficulty': 'easy'
        },
        {
            'input': 'A wave has a frequency of 50 Hz and a wavelength of 6 meters. What is its speed? (Use v = fλ)',
            'output': '300',
            'difficulty': 'medium'
        },
        {
            'input': 'What type of bond is formed when two atoms share electrons?',
            'output': 'Covalent bond',
            'difficulty': 'easy'
        },
        {
            'input': 'Calculate the kinetic energy of a 10 kg object moving at 5 m/s. (Use KE = ½mv²)',
            'output': '125',
            'difficulty': 'medium'
        },
        {
            'input': 'What is the molar mass of water (H₂O)? (H=1 g/mol, O=16 g/mol)',
            'output': '18',
            'difficulty': 'easy'
        },
        {
            'input': 'A gas occupies 5 L at 2 atm pressure. If the pressure increases to 5 atm at constant temperature, what is the new volume? (Use Boyle\'s Law: P₁V₁ = P₂V₂)',
            'output': '2',
            'difficulty': 'medium'
        },
        {
            'input': 'What is the powerhouse of the cell that produces ATP?',
            'output': 'Mitochondria',
            'difficulty': 'easy'
        }
    ]

    for q in science_questions:
        eval_data['domain'].append('science')
        eval_data['input'].append(q['input'])
        eval_data['output'].append(q['output'])
        eval_data['difficulty'].append(q['difficulty'])

    # ============================================================================
    # COMMONSENSE REASONING (10 questions)
    # ============================================================================
    commonsense_questions = [
        {
            'input': 'If you drop a glass cup on a concrete floor, what is most likely to happen?',
            'output': 'The glass cup will break/shatter',
            'difficulty': 'easy'
        },
        {
            'input': 'You see dark clouds gathering and hear thunder. What should you expect to happen soon?',
            'output': 'It will rain',
            'difficulty': 'easy'
        },
        {
            'input': 'If a restaurant has no customers during dinner time on a Friday night, what might this indicate about the restaurant?',
            'output': 'The restaurant might have poor quality food, bad service, or bad reputation',
            'difficulty': 'medium'
        },
        {
            'input': 'Someone is wearing a heavy winter coat, gloves, and a scarf in July. What might explain this unusual behavior?',
            'output': 'They might be in a cold location (southern hemisphere), have a medical condition, or be in a cold indoor environment',
            'difficulty': 'medium'
        },
        {
            'input': 'You arrive at work and notice the door is unlocked and the lights are already on, but you\'re usually the first one there. What should you consider?',
            'output': 'Someone else arrived early, someone forgot to lock up yesterday, or there might be an intruder',
            'difficulty': 'medium'
        },
        {
            'input': 'If ice cream is left out of the freezer for several hours, what will happen to it?',
            'output': 'It will melt',
            'difficulty': 'easy'
        },
        {
            'input': 'A person is running late for an important meeting and their car won\'t start. What would be the most practical immediate action?',
            'output': 'Call a taxi/rideshare, ask someone for a ride, or contact the meeting organizer',
            'difficulty': 'easy'
        },
        {
            'input': 'You notice your friend has been quieter than usual and avoiding social activities. What might this suggest?',
            'output': 'They might be dealing with personal issues, stress, depression, or going through a difficult time',
            'difficulty': 'medium'
        },
        {
            'input': 'If a plant\'s leaves are turning yellow and drooping, what are the most common causes?',
            'output': 'Overwatering, underwatering, lack of nutrients, or disease',
            'difficulty': 'medium'
        },
        {
            'input': 'Someone keeps looking at their watch during a conversation with you. What does this behavior typically indicate?',
            'output': 'They are in a hurry, have another appointment, or are not fully engaged in the conversation',
            'difficulty': 'easy'
        }
    ]

    for q in commonsense_questions:
        eval_data['domain'].append('commonsense_reasoning')
        eval_data['input'].append(q['input'])
        eval_data['output'].append(q['output'])
        eval_data['difficulty'].append(q['difficulty'])

    # ============================================================================
    # NUMERICAL REASONING (10 questions)
    # ============================================================================
    numerical_questions = [
        {
            'input': 'What is the next number in the sequence: 2, 4, 8, 16, 32, ?',
            'output': '64',
            'difficulty': 'easy'
        },
        {
            'input': 'What number should replace the question mark? 5 + 7 = 12, 9 + 11 = 20, 13 + ? = 28',
            'output': '15',
            'difficulty': 'easy'
        },
        {
            'input': 'If the pattern is: 1, 4, 9, 16, 25, what is the next number?',
            'output': '36',
            'difficulty': 'easy'
        },
        {
            'input': 'Complete the pattern: 100, 95, 85, 70, 50, ?',
            'output': '25',
            'difficulty': 'medium'
        },
        {
            'input': 'What is the missing number? 3, 6, 12, 24, ?, 96',
            'output': '48',
            'difficulty': 'easy'
        },
        {
            'input': 'Find the pattern: 2, 5, 11, 23, 47, ?',
            'output': '95',
            'difficulty': 'hard'
        },
        {
            'input': 'If A = 1, B = 2, C = 3, what is the numerical value of the word "CAB"?',
            'output': '6',
            'difficulty': 'easy'
        },
        {
            'input': 'What comes next: 1, 1, 2, 3, 5, 8, 13, ?',
            'output': '21',
            'difficulty': 'medium'
        },
        {
            'input': 'Complete: 1000, 500, 250, 125, ?',
            'output': '62.5',
            'difficulty': 'easy'
        },
        {
            'input': 'What number is missing? 7, 14, 28, ?, 112, 224',
            'output': '56',
            'difficulty': 'easy'
        }
    ]

    for q in numerical_questions:
        eval_data['domain'].append('numerical_reasoning')
        eval_data['input'].append(q['input'])
        eval_data['output'].append(q['output'])
        eval_data['difficulty'].append(q['difficulty'])

    # ============================================================================
    # FINANCIAL REASONING (10 questions)
    # ============================================================================
    financial_questions = [
        {
            'input': 'You invest $10,000 at 5% annual simple interest. How much interest will you earn after 3 years?',
            'output': '1500',
            'difficulty': 'easy'
        },
        {
            'input': 'A stock costs $50 per share. If it increases by 20% and then decreases by 20%, what is the final price?',
            'output': '48',
            'difficulty': 'medium'
        },
        {
            'input': 'If you save $200 per month for 2 years with no interest, how much will you have saved?',
            'output': '4800',
            'difficulty': 'easy'
        },
        {
            'input': 'A company\'s revenue is $500,000 and its expenses are $350,000. What is the profit margin percentage?',
            'output': '30',
            'difficulty': 'medium'
        },
        {
            'input': 'You take a loan of $5,000 at 8% annual interest. If you pay it back in one year, how much interest will you pay?',
            'output': '400',
            'difficulty': 'easy'
        },
        {
            'input': 'If a product is marked up by 40% from its cost of $25, what is the selling price?',
            'output': '35',
            'difficulty': 'easy'
        },
        {
            'input': 'An investment of $8,000 grows to $9,200 in one year. What is the annual return rate?',
            'output': '15',
            'difficulty': 'medium'
        },
        {
            'input': 'You buy 100 shares at $30 each and sell them at $35 each. What is your total profit before fees?',
            'output': '500',
            'difficulty': 'easy'
        },
        {
            'input': 'A bond pays 6% annual interest on a face value of $1,000. How much interest is paid per year?',
            'output': '60',
            'difficulty': 'easy'
        },
        {
            'input': 'If your monthly income is $4,500 and you spend 30% on rent, how much is left after rent?',
            'output': '3150',
            'difficulty': 'easy'
        }
    ]

    for q in financial_questions:
        eval_data['domain'].append('financial_reasoning')
        eval_data['input'].append(q['input'])
        eval_data['output'].append(q['output'])
        eval_data['difficulty'].append(q['difficulty'])

    # ============================================================================
    # READING COMPREHENSION (10 questions)
    # ============================================================================
    reading_questions = [
        {
            'input': 'Passage: "The Amazon rainforest produces 20% of Earth\'s oxygen. It spans 5.5 million square kilometers across nine countries, with Brazil containing the majority." Question: What percentage of Earth\'s oxygen does the Amazon produce?',
            'output': '20',
            'difficulty': 'easy'
        },
        {
            'input': 'Passage: "Marie Curie was the first woman to win a Nobel Prize and the only person to win Nobel Prizes in two different sciences - Physics in 1903 and Chemistry in 1911." Question: In which two fields did Marie Curie win Nobel Prizes?',
            'output': 'Physics and Chemistry',
            'difficulty': 'easy'
        },
        {
            'input': 'Passage: "The Great Wall of China stretches over 13,000 miles. Construction began in the 7th century BC and continued for centuries. Contrary to popular belief, it is not visible from space with the naked eye." Question: Can the Great Wall be seen from space with the naked eye?',
            'output': 'No',
            'difficulty': 'easy'
        },
        {
            'input': 'Passage: "Photosynthesis occurs in chloroplasts, converting CO2 and water into glucose and oxygen using sunlight. This process is crucial for life on Earth as it produces oxygen and forms the base of most food chains." Question: Where in plant cells does photosynthesis occur?',
            'output': 'Chloroplasts',
            'difficulty': 'easy'
        },
        {
            'input': 'Passage: "The Industrial Revolution began in Britain in the late 18th century. It transformed economies from agrarian to industrial, introducing factories, mass production, and urbanization. This period saw major innovations including the steam engine and mechanized textile production." Question: Where did the Industrial Revolution begin?',
            'output': 'Britain',
            'difficulty': 'easy'
        },
        {
            'input': 'Passage: "Honey never spoils. Archaeologists have found 3,000-year-old honey in Egyptian tombs that was still perfectly edible. Its low moisture content and acidic pH create an inhospitable environment for bacteria." Question: What two properties of honey prevent it from spoiling?',
            'output': 'Low moisture content and acidic pH',
            'difficulty': 'medium'
        },
        {
            'input': 'Passage: "The human brain contains approximately 86 billion neurons. Each neuron can form thousands of connections, creating a complex network responsible for all thoughts, memories, and behaviors." Question: How many neurons are in the human brain?',
            'output': '86 billion',
            'difficulty': 'easy'
        },
        {
            'input': 'Passage: "Shakespeare wrote 37 plays and 154 sonnets during his lifetime. His works have been translated into over 100 languages and continue to be performed more than any other playwright\'s." Question: How many sonnets did Shakespeare write?',
            'output': '154',
            'difficulty': 'easy'
        },
        {
            'input': 'Passage: "The speed of light in a vacuum is 299,792,458 meters per second, approximately 186,282 miles per second. Nothing can travel faster than light in a vacuum according to Einstein\'s theory of relativity." Question: What is the approximate speed of light in miles per second?',
            'output': '186,282',
            'difficulty': 'easy'
        },
        {
            'input': 'Passage: "Mount Everest stands at 8,849 meters (29,032 feet) above sea level, making it Earth\'s highest mountain. It was first successfully climbed by Edmund Hillary and Tenzing Norgay in 1953." Question: Who were the first people to successfully climb Mount Everest?',
            'output': 'Edmund Hillary and Tenzing Norgay',
            'difficulty': 'easy'
        }
    ]

    for q in reading_questions:
        eval_data['domain'].append('reading_comprehension')
        eval_data['input'].append(q['input'])
        eval_data['output'].append(q['output'])
        eval_data['difficulty'].append(q['difficulty'])

    # ============================================================================
    # CREATIVE WRITING (10 prompts)
    # ============================================================================
    creative_writing_prompts = [
        {
            'input': 'Write a short paragraph describing a mysterious abandoned lighthouse on a stormy night.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Create an opening sentence for a science fiction story about first contact with aliens.',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Write a dialogue between two old friends meeting after 20 years.',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Describe a character who discovers they have an unexpected superpower.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Write a haiku about autumn leaves.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Create a short story ending with the line: "And that was the last time anyone saw the old museum."',
            'output': None,
            'difficulty': 'hard'
        },
        {
            'input': 'Write a letter from a time traveler to their past self.',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Describe a futuristic city from the perspective of someone visiting it for the first time.',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Write a conversation between the sun and the moon.',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Create a suspenseful opening paragraph for a thriller novel.',
            'output': None,
            'difficulty': 'hard'
        }
    ]

    for q in creative_writing_prompts:
        eval_data['domain'].append('creative_writing')
        eval_data['input'].append(q['input'])
        eval_data['output'].append(q['output'])
        eval_data['difficulty'].append(q['difficulty'])

    # ============================================================================
    # CREATIVE IDEATION (10 prompts)
    # ============================================================================
    creative_ideation_prompts = [
        {
            'input': 'Generate 3 innovative uses for old smartphones that can no longer run modern apps.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Suggest 5 creative names for a coffee shop that specializes in sustainable and eco-friendly practices.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Come up with 3 app ideas that could help reduce food waste in households.',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Brainstorm 4 unique birthday party themes for a 10-year-old who loves science.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Generate 3 innovative solutions to reduce plastic use in grocery stores.',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Suggest 5 creative team-building activities for remote workers.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Come up with 3 innovative features for a smartwatch designed specifically for elderly users.',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Brainstorm 4 creative ways to repurpose old furniture instead of throwing it away.',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Generate 3 unique marketing campaign ideas for promoting mental health awareness among teenagers.',
            'output': None,
            'difficulty': 'hard'
        },
        {
            'input': 'Suggest 5 innovative classroom activities that combine art and mathematics.',
            'output': None,
            'difficulty': 'medium'
        }
    ]

    for q in creative_ideation_prompts:
        eval_data['domain'].append('creative_ideation')
        eval_data['input'].append(q['input'])
        eval_data['output'].append(q['output'])
        eval_data['difficulty'].append(q['difficulty'])

    # ============================================================================
    # SUMMARIZATION (10 passages)
    # ============================================================================
    summarization_tasks = [
        {
            'input': 'Summarize this passage in 2 sentences: "Climate change is causing global temperatures to rise at an unprecedented rate. The primary driver is greenhouse gas emissions from human activities, particularly the burning of fossil fuels for energy. This warming leads to melting ice caps, rising sea levels, more frequent extreme weather events, and disruptions to ecosystems worldwide. Scientists warn that without significant action to reduce emissions, the consequences will become increasingly severe."',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Summarize the key points: "Artificial Intelligence is transforming healthcare through early disease detection, personalized treatment plans, and drug discovery. AI algorithms can analyze medical images more accurately than humans in some cases, identifying cancers and other conditions at earlier stages. Machine learning models can predict patient outcomes and suggest optimal treatment approaches based on vast amounts of medical data. However, challenges remain around data privacy, algorithmic bias, and the need for human oversight in medical decision-making."',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Provide a brief summary: "The ancient Egyptians built the pyramids as tombs for their pharaohs and queens. The Great Pyramid of Giza, built around 2560 BC, is the largest and oldest of the three pyramids in the Giza pyramid complex. It took approximately 20 years to build and required the labor of thousands of workers. The pyramids showcase the Egyptians\' advanced knowledge of mathematics, astronomy, and engineering."',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Summarize in 1-2 sentences: "Social media platforms have fundamentally changed how people communicate, share information, and consume news. While these platforms enable instant global connectivity and democratize information sharing, they also face criticism for spreading misinformation, creating echo chambers, and negatively impacting mental health, particularly among young users."',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Summarize the main idea: "Renewable energy sources like solar and wind power are becoming increasingly cost-competitive with fossil fuels. Technological advances have dramatically reduced the cost of solar panels and wind turbines over the past decade. Many countries are now investing heavily in renewable energy infrastructure to meet climate goals and ensure energy security. However, challenges remain in energy storage and grid integration to handle the intermittent nature of renewable sources."',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Provide a concise summary: "The human gut microbiome consists of trillions of microorganisms that play crucial roles in digestion, immune function, and even mental health. Recent research suggests that the composition of gut bacteria can influence mood, behavior, and susceptibility to various diseases. Diet, antibiotics, and lifestyle factors significantly impact microbiome diversity. Scientists are exploring how manipulating the microbiome through probiotics and dietary interventions might treat various health conditions."',
            'output': None,
            'difficulty': 'hard'
        },
        {
            'input': 'Summarize this text: "Electric vehicles (EVs) are rapidly gaining market share as battery technology improves and charging infrastructure expands. Modern EVs offer ranges comparable to gasoline vehicles, with lower operating costs and zero direct emissions. Governments worldwide are implementing incentives to accelerate EV adoption. The main challenges include battery production environmental impact, charging time, and initial purchase price."',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Provide a brief summary: "The Internet of Things (IoT) refers to the network of physical devices embedded with sensors and software that connect and exchange data. IoT applications range from smart home devices and wearable fitness trackers to industrial sensors and smart city infrastructure. While IoT promises increased efficiency and convenience, it also raises concerns about data security, privacy, and the potential for large-scale cyberattacks."',
            'output': None,
            'difficulty': 'medium'
        },
        {
            'input': 'Summarize the key points: "Sleep is essential for physical and mental health, yet many people don\'t get enough quality sleep. Adults typically need 7-9 hours per night for optimal functioning. Chronic sleep deprivation is linked to numerous health problems including obesity, diabetes, cardiovascular disease, and mental health issues. Good sleep hygiene practices include maintaining a consistent sleep schedule, avoiding screens before bed, and creating a comfortable sleep environment."',
            'output': None,
            'difficulty': 'easy'
        },
        {
            'input': 'Summarize in 2-3 sentences: "Quantum computing represents a revolutionary approach to computation, using quantum mechanical phenomena to perform calculations impossible for classical computers. While still in early stages, quantum computers could potentially break current encryption methods, accelerate drug discovery, optimize complex systems, and solve previously intractable problems. Major tech companies and governments are investing billions in quantum computing research, though practical, large-scale quantum computers remain years away."',
            'output': None,
            'difficulty': 'hard'
        }
    ]

    for q in summarization_tasks:
        eval_data['domain'].append('summarization')
        eval_data['input'].append(q['input'])
        eval_data['output'].append(q['output'])
        eval_data['difficulty'].append(q['difficulty'])

    # Create DataFrame
    eval_df = pd.DataFrame(eval_data)
    
    return eval_df


Multi-Metric Domain-Aware Evaluation


In [62]:

def get_gemma_answer(valid_df,batch=25,waiting=30):

    gemma_rows = []
    i=1
    for idx, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc="Gemma Base Model Responses"):
        prompt = row['input']
        try:
            gemma_resp = generate_gemma_response(prompt=prompt)
        except Exception as e:
            gemma_resp = None
            print(f"Gemma error idx {idx}: {e}")

        gemma_rows.append({
            'uid': row['uid'],
            'input': row['input'],
            'gemma_response': gemma_resp
        })
        
        if i%batch==0:
            print(f"Waiting for {waiting} seconds")
            time.sleep(waiting)
        i=i+1
    gemma_df = pd.DataFrame(gemma_rows)
    return gemma_df


def evaluate_base_model_with_ground_truth(input_text, model_response, ground_truth, domain):
    '''Evaluate response against ground truth with 4 separate scores'''
    
    prompt = f'''
Evaluate this AI model's response for a {domain} task with scores:

Question/Task: {input_text}

Ground Truth Answer: {ground_truth}

Model Response: {model_response}

Provide 2 separate scores (each 1-10):


1. ANSWER QUALITY (1-10):
   - Is the answer well-formatted and complete?
   - Is it clear and understandable?
   - Does it directly address the question?

2. ANSWER ACCURACY (0 = Wrong/1 =Correct):
   - Does the final answer match the ground truth?
   - Is it mathematically/factually correct?
   - If the answer is correct but is represented in a different format it's fine


Provide your evaluation in this exact format:
Answer Accuracy: 0/1
ANSWER QUALITY: X/10
Feedback: [2-3 sentences explaining the scores]
'''
    
    try:
        response = gemini_model.generate_content(prompt)
        return parse_scores(response.text)
    except Exception as e:
        print(f"Error evaluating: {e}")
        return None


def evaluate_base_model_creative(input_text, model_response, domain):
    '''Evaluate creative/generative responses with 3 scores (no accuracy)'''
    
    domain_criteria = {
        'code': 'Code correctness, efficiency, handles edge cases, solves the problem',
        'creative_ideation': 'Creativity, relevance to prompt, feasibility, originality',
        'creative_writing': 'Writing quality, coherence, creativity, engagement',
        'summarization': 'Accuracy of main points, completeness, conciseness, clarity'
    }
    
    criteria = domain_criteria.get(domain, 'Quality, relevance, completeness, creativity')
    
    prompt = f'''
Evaluate this AI model's response for a {domain} task with scores:

Task/Prompt: {input_text}

Model Response: {model_response}

Provide score (1-10):


1. ANSWER QUALITY (1-10):
   - {criteria}
   - Is the answer complete and well-executed?
   - Does it fulfill the prompt requirements?

Note: No accuracy score for {domain} as there's no single correct answer.

Provide your evaluation in this exact format:

Answer Accuracy: N/A
ANSWER QUALITY: X/10
Feedback: [2-3 sentences explaining the scores]
'''
    
    try:
        response = gemini_model.generate_content(prompt)
        scores = parse_scores(response.text)
        if scores:
            scores['answer_accuracy'] = None  # N/A for creative domains
        return scores
    except Exception as e:
        print(f"Error evaluating: {e}")
        return None
    
    
def evaluate_gemma_base_model(eval_df):
    # Run multi-metric evaluation
    print("\nEvaluating responses with multi-metric scoring...")

    answer_quality_scores = []
    answer_accuracy_scores = []
    feedbacks = []

    # has_output = 'output' in eval_df.columns

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Multi-metric judging"):
        # Skip failed generations
        # if row['error'] is not None:
        #     format_scores.append(0)
        #     reasoning_scores.append(0)
        #     answer_quality_scores.append(0)
        #     answer_accuracy_scores.append(0)
        #     feedbacks.append("Generation failed")
        #     continue
        
        domain = row['domain']
        
        # Determine evaluation strategy
        if domain in GROUND_TRUTH_DOMAINS:
            scores = evaluate_base_model_with_ground_truth(
                input_text=row['input'],
                model_response=row['gemma_response'],
                ground_truth=row['ground_truth'],
                domain=domain
            )
        elif domain in CREATIVE_DOMAINS:
            scores = evaluate_base_model_creative(
                input_text=row['input'],
                model_response=row['gemma_response'],
                domain=domain
            )
        else:
            scores = evaluate_base_model_creative(
                input_text=row['input'],
                model_response=row['gemma_response'],
                domain=domain
            )
        
        if scores:
            answer_quality_scores.append(scores.get('answer_quality', 0))
            answer_accuracy_scores.append(scores.get('answer_accuracy', 0) if scores.get('answer_accuracy') is not None else None)
            feedbacks.append(scores.get('feedback', 'Evaluation completed'))
        else:
            answer_quality_scores.append(0)
            answer_accuracy_scores.append(None)
            feedbacks.append("Evaluation failed")
        
        # Rate limiting
        time.sleep(0.5)

    # Add to dataframe

    eval_df['answer_quality'] = answer_quality_scores
    eval_df['answer_accuracy'] = answer_accuracy_scores
    eval_df['judge_feedback'] = feedbacks

    print(f"  Answer Quality: {np.mean([s for s in answer_quality_scores if s > 0]):.2f}/10")

    accuracy_valid = [s for s in answer_accuracy_scores if s is not None and s > 0]
    if accuracy_valid:
        print(f"  Answer Accuracy (verifiable domains): {np.mean(accuracy_valid):.2f}/10")

    # Per-domain statistics
    print("\nPer-Domain Scores:")
    for domain in eval_df['domain'].unique():
        domain_df = eval_df[eval_df['domain'] == domain]
        answer_quality_d = [s for s in domain_df['answer_quality'] if s > 0]
       
        if answer_quality_d:
            print(f"\n  {domain}:")
            print(f"    Answer Quality: {np.mean(answer_quality_d):.2f}/10")
            
            if domain in GROUND_TRUTH_DOMAINS:
                domain_acc = [s for s in domain_df['answer_accuracy'] if s is not None and s > 0]
                if domain_acc:
                    print(f"    Accuracy: {np.mean(domain_acc):.2f}/10")

    return eval_df

## Evaluating validation sample

- Evaluating the tuned model

In [36]:
validation_sample = pd.read_csv("..//evaluation//eval_data//eval_results_v3.csv")
valid_eval = evaluate_model(eval_df=validation_sample)


Evaluating responses with multi-metric scoring...


Multi-metric judging:   0%|          | 0/160 [00:00<?, ?it/s]

Multi-metric judging: 100%|██████████| 160/160 [23:29<00:00,  8.81s/it]


✓ Multi-metric evaluation complete

Overall Statistics:
  Format Accuracy: 9.01/10
  Reasoning Quality: 6.35/10
  Answer Quality: 7.15/10
  Answer Accuracy (verifiable domains): 1.00/10
  Overall Weighted Score: 6.92/10

Per-Domain Scores:

  code:
    Overall: 6.57/10
    Format: 10.00/10
    Reasoning: 5.90/10

  math:
    Overall: 6.86/10
    Format: 9.60/10
    Reasoning: 6.80/10
    Accuracy: 1.00/10

  commonsense_reasoning:
    Overall: 7.31/10
    Format: 9.00/10
    Reasoning: 5.55/10
    Accuracy: 1.00/10

  creative_ideation:
    Overall: 7.87/10
    Format: 4.35/10
    Reasoning: 8.40/10

  creative_writing:
    Overall: 8.36/10
    Format: 10.00/10
    Reasoning: 8.30/10

  financial_reasoning:
    Overall: 4.66/10
    Format: 10.00/10
    Reasoning: 4.89/10
    Accuracy: 1.00/10

  numerical_reasoning:
    Overall: 4.13/10
    Format: 9.90/10
    Reasoning: 4.10/10
    Accuracy: 1.00/10

  reading_comprehension:
    Overall: 6.67/10
    Format: 9.60/10
    Reasoning: 5.3

- Evaluation of base model gemma

    - Get responses for validation sample from gemma
    - Evaluate the answers 

In [63]:

gemma_df = get_gemma_answer(valid_df=validation_sample[['uid','input','ground_truth','domain']])
gemma_df = pd.merge(gemma_df,validation_sample[['uid','domain','ground_truth']])

Gemma Base Model Responses:  15%|█▌        | 24/160 [00:48<02:45,  1.21s/it]

Waiting for 30 seconds


Gemma Base Model Responses:  31%|███       | 49/160 [02:09<07:20,  3.96s/it]

Waiting for 30 seconds


Gemma Base Model Responses:  46%|████▋     | 74/160 [04:06<03:42,  2.58s/it]

...

Gemma Base Model Responses:  78%|███████▊  | 124/160 [06:01<00:38,  1.07s/it]

Waiting for 30 seconds


Gemma Base Model Responses:  93%|█████████▎| 149/160 [07:05<00:16,  1.50s/it]

Waiting for 30 seconds


Gemma Base Model Responses: 100%|██████████| 160/160 [07:50<00:00,  2.94s/it]


In [64]:
gemma_eval=evaluate_gemma_base_model(gemma_df)


Evaluating responses with multi-metric scoring...


Multi-metric judging: 100%|██████████| 160/160 [17:45<00:00,  6.66s/it]

  Answer Quality: 7.40/10
  Answer Accuracy (verifiable domains): 1.00/10

Per-Domain Scores:

  code:
    Answer Quality: 6.80/10

  math:
    Answer Quality: 7.90/10
    Accuracy: 1.00/10

  commonsense_reasoning:
    Answer Quality: 8.75/10
    Accuracy: 1.00/10

  creative_ideation:
    Answer Quality: 9.53/10

  creative_writing:
    Answer Quality: 7.60/10

  financial_reasoning:
    Answer Quality: 6.30/10
    Accuracy: 1.00/10

  numerical_reasoning:
    Answer Quality: 4.80/10

  reading_comprehension:
    Answer Quality: 7.35/10
    Accuracy: 1.00/10

  science:
    Answer Quality: 6.70/10
    Accuracy: 1.00/10

  summarization:
    Answer Quality: 6.35/10


In [77]:
valid_eval

,uid,input,ground_truth,domain,problem_type,reasoning,response,model_response,error,format_accuracy,reasoning_quality,answer_quality,answer_accuracy,judge_feedback,overall_score
0,code18321,How can we check whether a given number is pri...,NaN,code,code generation,"We need to answer: explain solution approach, ...",**Solution Approach**\n\n1. **Handle edge case...,<reasoning>We need to explain approach and giv...,NaN,10.0,2.0,1.0,NaN,"The reasoning is confused, proposing an algori...",2.70
1,code54518,Below is an instruction that describes a task....,NaN,code,python code generation,"The user asks: ""Below is an instruction that d...",**Solution Overview**\n\nTo obtain the maximum...,<reasoning>We need to produce explanation and ...,NaN,10.0,6.0,9.0,NaN,The model followed the formatting instructions...,8.10
2,code51476,Below is an instruction that describes a task....,NaN,code,python code generation,We need to respond with a solution that meets ...,**Solution Approach**\n\n1. **Load the dataset...,"<reasoning>The user wants a response that ""app...",NaN,10.0,9.0,8.0,NaN,The model demonstrates excellent adherence to ...,8.65
3,code17404,Edit the code to allow users to enter their na...,NaN,code,code generation,We need to edit code to allow user input for n...,**Solution Overview**\n\n1. **Read input** – U...,<reasoning>We need to edit the code so that us...,NaN,10.0,9.0,9.0,NaN,The model delivered a very well-structured res...,9.15
4,code25223,What is the output of the following Java code?...,NaN,code,code generation,We need to answer: What is output? The code de...,**Explanation**\n\n- `x % y` is the **modulo (...,<reasoning>We need to parse the code. It's a s...,NaN,10.0,10.0,10.0,NaN,The model correctly uses and closes both the `...,10.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,summarization535,Summarize the following news article:\n\nWASHI...,Justice Department must decide whether to purs...,summarization,news_summary,We need to produce concise summary 100-200 wor...,The CIA has asked the Justice Department to re...,<reasoning>We need to summarize the news artic...,NaN,9.0,3.0,4.0,NaN,The model correctly used the `<reasoning>` and...,4.40
156,summarization749,Summarize the following news article:\n\n(CNN)...,Sri Lankan army Web site: Two heavily fortifie...,summarization,news_summary,We need to produce a concise summary 100-200 w...,Sri Lankan forces announced on Tuesday that th...,<reasoning>We need to summarize the news artic...,NaN,10.0,6.0,5.0,NaN,"The format is impeccable, with correct tag usa...",6.10
157,summarization1325,Summarize the following news article:\n\n(Ente...,Entertainment Weekly reviews Iain Softley's ad...,summarization,news_summary,"We need to summarize the article, 100-200 word...",Entertainment Weekly’s review of Iain Softley’...,"<reasoning>We need to summarize the article, e...",NaN,10.0,4.0,3.0,NaN,The model correctly implemented the required f...,4.40
158,summarization1517,Summarize the following news article:\n\n(CNN)...,Christopher Lloyd's home one of dozens destroy...,summarization,news_summary,We need to produce concise summary 100-200 wor...,Actor Christopher Lloyd surveyed the wreckage ...,"<reasoning>We need to summarize the article, t...",NaN,10.0,9.0,10.0,NaN,The model flawlessly used the specified format...,9.65


- Evaluating Teacher Response

In [79]:
teacher_valid=validation_sample[['uid','input','domain','reasoning','response','ground_truth']]
def format_output(reasoning, response):
    reasoning = str(reasoning) if not pd.isna(reasoning) else ""
    response = str(response) if not pd.isna(response) else ""
    return f"<reasoning>{reasoning.strip()}</reasoning><answer>{response.strip()}</answer>"

teacher_valid['model_response'] = teacher_valid.apply(
    lambda row: format_output(row['reasoning'], row['response']),
    axis=1
)
teacher_valid=teacher_valid.drop(columns={'reasoning','response'})
valid_teacher_eval = evaluate_model(eval_df=teacher_valid)
valid_teacher_eval.rename(
    columns={'answer_quality':'teacher_answer_quality','answer_accuracy':'teacher_answer_accuracy'
             ,'format_accuracy':'teacher_format_accuracy','reasoning_quality':'teacher_reasoning_quality'},
    inplace=True
)

C:\Users\hazar\AppData\Local\Temp\ipykernel_16564\2121915070.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  teacher_valid['model_response'] = teacher_valid.apply(



Evaluating responses with multi-metric scoring...


Multi-metric judging:   0%|          | 0/160 [00:00<?, ?it/s]

Multi-metric judging: 100%|██████████| 160/160 [16:20<00:00,  6.13s/it]


✓ Multi-metric evaluation complete

Overall Statistics:
  Format Accuracy: 9.91/10
  Reasoning Quality: 8.79/10
  Answer Quality: 9.64/10
  Answer Accuracy (verifiable domains): 1.00/10
  Overall Weighted Score: 9.37/10

Per-Domain Scores:

  code:
    Overall: 9.44/10
    Format: 10.00/10
    Reasoning: 8.40/10

  math:
    Overall: 9.93/10
    Format: 9.70/10
    Reasoning: 9.80/10
    Accuracy: 1.00/10

  commonsense_reasoning:
    Overall: 9.58/10
    Format: 9.85/10
    Reasoning: 8.15/10
    Accuracy: 1.00/10

  creative_ideation:
    Overall: 9.32/10
    Format: 10.00/10
    Reasoning: 9.05/10

  creative_writing:
    Overall: 9.21/10
    Format: 10.00/10
    Reasoning: 9.25/10

  financial_reasoning:
    Overall: 8.88/10
    Format: 10.00/10
    Reasoning: 9.70/10
    Accuracy: 1.00/10

  numerical_reasoning:
    Overall: 9.29/10
    Format: 9.90/10
    Reasoning: 9.10/10
    Accuracy: 1.00/10

  reading_comprehension:
    Overall: 9.39/10
    Format: 9.95/10
    Reasoning: 8.

In [80]:

gemma_eval.rename(
    columns={'answer_quality':'gemma_answer_quality','answer_accuracy':'gemma_answer_accuracy'},
    inplace=True
)
final_combined_df = pd.merge(valid_eval,gemma_eval[['uid','gemma_response','gemma_answer_quality','gemma_answer_accuracy']],
                             on=['uid'],how='inner')
final_combined_df = pd.merge(final_combined_df,valid_teacher_eval[['uid','teacher_format_accuracy','teacher_reasoning_quality'
                                                                   ,'teacher_answer_quality','teacher_answer_accuracy']])


In [82]:
def compare_model_performance(final_combined_df):
    """Compare tuned model vs Gemma base vs Teacher model performance"""
    
    results = {}
    
    # Overall Statistics
    print("="*60)
    print("OVERALL PERFORMANCE COMPARISON")
    print("="*60)
    
    # Tuned Model
    tuned_quality = final_combined_df['answer_quality'].dropna()
    tuned_accuracy = final_combined_df['answer_accuracy'].dropna()
    tuned_reasoning = final_combined_df['reasoning_quality'].dropna()
    tuned_format = final_combined_df['format_accuracy'].dropna()
    
    # Gemma Base Model
    gemma_quality = final_combined_df['gemma_answer_quality'].dropna()
    gemma_accuracy = final_combined_df['gemma_answer_accuracy'].dropna()
    
    # Teacher Model
    teacher_quality = final_combined_df['teacher_answer_quality'].dropna()
    teacher_accuracy = final_combined_df['teacher_answer_accuracy'].dropna()
    teacher_reasoning = final_combined_df['teacher_reasoning_quality'].dropna()
    teacher_format = final_combined_df['teacher_format_accuracy'].dropna()
    
    # Store results
    results['tuned'] = {
        'quality_mean': tuned_quality.mean(),
        'quality_std': tuned_quality.std(),
        'accuracy_mean': tuned_accuracy.mean(),
        'accuracy_std': tuned_accuracy.std(),
        'reasoning_mean': tuned_reasoning.mean(),
        'reasoning_std': tuned_reasoning.std(),
        'format_mean': tuned_format.mean(),
        'format_std': tuned_format.std(),
        'n_samples': len(tuned_quality)
    }
    
    results['gemma'] = {
        'quality_mean': gemma_quality.mean(),
        'quality_std': gemma_quality.std(),
        'accuracy_mean': gemma_accuracy.mean(),
        'accuracy_std': gemma_accuracy.std(),
        'n_samples': len(gemma_quality)
    }
    
    results['teacher'] = {
        'quality_mean': teacher_quality.mean(),
        'quality_std': teacher_quality.std(),
        'accuracy_mean': teacher_accuracy.mean(),
        'accuracy_std': teacher_accuracy.std(),
        'reasoning_mean': teacher_reasoning.mean(),
        'reasoning_std': teacher_reasoning.std(),
        'format_mean': teacher_format.mean(),
        'format_std': teacher_format.std(),
        'n_samples': len(teacher_quality)
    }
    
    # Print comprehensive comparison
    print(f"\n{'Model':<15} {'Format Acc':<20} {'Reasoning Q':<20} {'Answer Q':<20} {'Answer Acc':<20} {'n':<8}")
    print("-"*110)
    print(f"{'Teacher':<15} {teacher_format.mean():>5.2f} ± {teacher_format.std():<4.2f}       {teacher_reasoning.mean():>5.2f} ± {teacher_reasoning.std():<4.2f}      {teacher_quality.mean():>5.2f} ± {teacher_quality.std():<4.2f}      {teacher_accuracy.mean():>5.2f} ± {teacher_accuracy.std():<4.2f}      {len(teacher_quality):<8}")
    print(f"{'Tuned':<15} {tuned_format.mean():>5.2f} ± {tuned_format.std():<4.2f}       {tuned_reasoning.mean():>5.2f} ± {tuned_reasoning.std():<4.2f}      {tuned_quality.mean():>5.2f} ± {tuned_quality.std():<4.2f}      {tuned_accuracy.mean():>5.2f} ± {tuned_accuracy.std():<4.2f}      {len(tuned_quality):<8}")
    print(f"{'Gemma Base':<15} {'N/A':<20} {'N/A':<20} {gemma_quality.mean():>5.2f} ± {gemma_quality.std():<4.2f}      {gemma_accuracy.mean():>5.2f} ± {gemma_accuracy.std():<4.2f}      {len(gemma_quality):<8}")
    
    # Gaps from ceiling
    format_gap = teacher_format.mean() - tuned_format.mean()
    reasoning_gap = teacher_reasoning.mean() - tuned_reasoning.mean()
    quality_gap = teacher_quality.mean() - tuned_quality.mean()
    accuracy_gap = teacher_accuracy.mean() - tuned_accuracy.mean()
    
    print(f"\nGap to Teacher Ceiling:")
    print(f"  Format Accuracy: {format_gap:.2f} points ({(format_gap/teacher_format.mean())*100:.1f}% below ceiling)")
    print(f"  Reasoning Quality: {reasoning_gap:.2f} points ({(reasoning_gap/teacher_reasoning.mean())*100:.1f}% below ceiling)")
    print(f"  Answer Quality: {quality_gap:.2f} points ({(quality_gap/teacher_quality.mean())*100:.1f}% below ceiling)")
    print(f"  Answer Accuracy: {accuracy_gap:.2f} points ({(accuracy_gap/teacher_accuracy.mean())*100:.1f}% below ceiling)")
    
    # Improvement over base
    quality_improvement = ((tuned_quality.mean() - gemma_quality.mean()) / gemma_quality.mean()) * 100
    accuracy_improvement = ((tuned_accuracy.mean() - gemma_accuracy.mean()) / gemma_accuracy.mean()) * 100
    
    print(f"\nImprovement over Gemma Base:")
    print(f"  Answer Quality: {quality_improvement:+.1f}%")
    print(f"  Answer Accuracy: {accuracy_improvement:+.1f}%")
    
    # Per-Domain Breakdown
    print("\n" + "="*60)
    print("PER-DOMAIN COMPARISON")
    print("="*60)
    
    for domain in final_combined_df['domain'].unique():
        domain_df = final_combined_df[final_combined_df['domain'] == domain]
        
        print(f"\n{domain.upper()}:")
        
        t_quality = domain_df['answer_quality'].dropna()
        t_accuracy = domain_df['answer_accuracy'].dropna()
        t_reasoning = domain_df['reasoning_quality'].dropna()
        t_format = domain_df['format_accuracy'].dropna()
        
        g_quality = domain_df['gemma_answer_quality'].dropna()
        g_accuracy = domain_df['gemma_answer_accuracy'].dropna()
        
        teach_quality = domain_df['teacher_answer_quality'].dropna()
        teach_accuracy = domain_df['teacher_answer_accuracy'].dropna()
        teach_reasoning = domain_df['teacher_reasoning_quality'].dropna()
        teach_format = domain_df['teacher_format_accuracy'].dropna()
        
        if len(teach_format) > 0:
            print(f"  Format:    Teacher={teach_format.mean():.2f}  Tuned={t_format.mean():.2f}  Gap={teach_format.mean()-t_format.mean():.2f}")
        
        if len(teach_reasoning) > 0:
            print(f"  Reasoning: Teacher={teach_reasoning.mean():.2f}  Tuned={t_reasoning.mean():.2f}  Gap={teach_reasoning.mean()-t_reasoning.mean():.2f}")
        
        if len(teach_quality) > 0:
            print(f"  Quality:   Teacher={teach_quality.mean():.2f}  Tuned={t_quality.mean():.2f}  Gemma={g_quality.mean():.2f}  Gap={teach_quality.mean()-t_quality.mean():.2f}")
        
        if len(teach_accuracy) > 0:
            print(f"  Accuracy:  Teacher={teach_accuracy.mean():.2f}  Tuned={t_accuracy.mean():.2f}  Gemma={g_accuracy.mean():.2f}  Gap={teach_accuracy.mean()-t_accuracy.mean():.2f}")
        
        results[domain] = {
            'teacher_format': teach_format.mean() if len(teach_format) > 0 else None,
            'tuned_format': t_format.mean() if len(t_format) > 0 else None,
            'teacher_reasoning': teach_reasoning.mean() if len(teach_reasoning) > 0 else None,
            'tuned_reasoning': t_reasoning.mean() if len(t_reasoning) > 0 else None,
            'teacher_quality': teach_quality.mean() if len(teach_quality) > 0 else None,
            'tuned_quality': t_quality.mean() if len(t_quality) > 0 else None,
            'gemma_quality': g_quality.mean() if len(g_quality) > 0 else None,
            'teacher_accuracy': teach_accuracy.mean() if len(teach_accuracy) > 0 else None,
            'tuned_accuracy': t_accuracy.mean() if len(t_accuracy) > 0 else None,
            'gemma_accuracy': g_accuracy.mean() if len(g_accuracy) > 0 else None,
        }
    
    return results

stats = compare_model_performance(final_combined_df)

OVERALL PERFORMANCE COMPARISON

Model           Format Acc           Reasoning Q          Answer Q             Answer Acc           n       
--------------------------------------------------------------------------------------------------------------
Teacher          9.91 ± 0.29        8.79 ± 1.83       9.64 ± 0.72       0.94 ± 0.23      160     
Tuned            9.01 ± 2.38        6.31 ± 3.25       7.11 ± 2.87       0.63 ± 0.48      160     
Gemma Base      N/A                  N/A                   7.40 ± 2.85       0.56 ± 0.50      160     

Gap to Teacher Ceiling:
  Format Accuracy: 0.89 points (9.0% below ceiling)
  Reasoning Quality: 2.48 points (28.2% below ceiling)
  Answer Quality: 2.54 points (26.3% below ceiling)
  Answer Accuracy: 0.31 points (32.9% below ceiling)

Improvement over Gemma Base:
  Answer Quality: -3.9%
  Answer Accuracy: +14.0%

PER-DOMAIN COMPARISON

CODE:
  Format:    Teacher=10.00  Tuned=10.00  Gap=0.00
  Reasoning: Teacher=8.40  Tuned=5.90  Gap=2.50
  Qu

In [83]:
final_combined_df.to_excel('..//evaluation/results/validation_eval_comp_modelrunV3.xlsx')

In [86]:
chk=chk[chk['domain'].isin(['summarization','create_ideation'])]
chk

,uid,input,domain,reasoning,response,ground_truth
140,summarization1524,Summarize the following news article:\n\n(CNN)...,summarization,We need to produce concise summary 100-200 wor...,Investigators with the U.S. Food and Drug Admi...,FDA: Tomatoes suspected in salmonella outbreak...
141,summarization585,Summarize the following news article:\n\n(CNN)...,summarization,We need to produce a concise summary 100-200 w...,"In an edited CNN interview, Donald Trump—busin...",Donald Trump says executives should have caps ...
142,summarization1902,Summarize the following news article:\n\n(CNN)...,summarization,We need to produce concise summary 100-200 wor...,Jose Mourinho has signed a one‑year extension ...,Jose Mourinho has extended his contract with S...
143,summarization623,Summarize the following news article:\n\n(CNN)...,summarization,"The user wants a summary of the news article, ...",President Obama announced that the first infra...,New highway resurfacing project in Maryland is...
144,summarization52,Summarize the following news article:\n\nMIAMI...,summarization,We need to produce concise summary 100-200 wor...,A federal kidnapping plot in Broward County wa...,"Woman, boyfriend, third man charged in bizarre..."
145,summarization1582,Summarize the following news article:\n\nATLAN...,summarization,We need to produce a concise summary 100-200 w...,"Samuel Welsh, a 29‑year‑old with spina bifida ...",Institute that places disabled employees says ...
146,summarization1213,Summarize the following news article:\n\nLOS A...,summarization,We need to produce a concise summary 100-200 w...,Liv Tyler says she was surprised to get a late...,"Actress Liv Tyler says ""Hulk"" role came unexpe..."
147,summarization2109,Summarize the following news article:\n\nLONDO...,summarization,We need to produce concise summary 100-200 wor...,"Valerie Gooding, chief executive of global hea...","Valerie Gooding, CEO of BUPA, speaks to CNN's ..."
148,summarization2116,Summarize the following news article:\n\nNEW Y...,summarization,We need to produce a concise summary 100-200 w...,U.S. authorities arrested ten members of a tra...,Officials say the original versions of the goo...
149,summarization1926,Summarize the following news article:\n\nATLAN...,summarization,We need to summarize the news article. The art...,Jeremy Piven stopped by CNN’s newsroom for an ...,Jeremy Piven plays a used car salesman in movi...


In [92]:
chk=validation_sample[['uid','input','domain','reasoning','response','ground_truth']]
chk=chk[chk['domain'].isin(['summarization','creative_ideation'])]
def format_output(reasoning, response):
    reasoning = str(reasoning) if not pd.isna(reasoning) else ""
    response = str(response) if not pd.isna(response) else ""
    return f"<reasoning>{reasoning.strip()}</reasoning><answer>{response.strip()}</answer>"

chk['model_response'] = chk.apply(
    lambda row: format_output(row['reasoning'], row['response']),
    axis=1
)


def print_samples(df):
    
    samp=df.sample(1)
    print(f"Reasoning:{samp['reasoning'].values[0]}")
    print(f"Response:{samp['response'].values[0]}")
    print(f"Formatted:{samp['model_response'].values[0]}")

In [97]:
print_samples(df=chk)

Reasoning:We need to brainstorm 5 ways to improve education system for non-technical users, prioritize accessibility. Provide specific, actionable, innovative solutions, under 300-600 words. Must be creative, practical, thorough. We'll output a list of 5 ideas with brief description and steps. Ensure under 600 words.
Response:**5 Action‑Oriented Ways to Make Education More Accessible for Non‑Technical Learners**

| # | Idea (One‑Sentence Pitch) | How It Works (Key Steps) | Why It Boosts Accessibility |
|---|---------------------------|--------------------------|------------------------------|
| **1** | **Universal “Learning‑First” Platform** – a cloud‑based hub that auto‑adjusts content format (text, audio, video, captions, sign‑language overlay) based on each learner’s device, bandwidth, and preferred modality. | 1. Deploy a lightweight web app that detects screen‑reader, high‑contrast, and bandwidth settings. <br>2. Store every lesson in a **content‑agnostic markup** (e.g., CommonMar

## Validation on Claude Base Sample for evaluation

In [ ]:
general_eval_from_claude_df = generate_general_eval_from_claude()



Evaluating responses with multi-metric scoring...


Multi-metric judging:   0%|          | 0/100 [00:00<?, ?it/s]


KeyError: 'model_response'

In [ ]:
valid_eval.to_excel('../evaluation//results//validation_eval_modelrunV3.xlsx')
valid_claude_gen_eval.to_excel('../evaluation//results//validation_claudegen_modelrunV3.xlsx')